In [ ]:
%pip install -q otter-grader

# Case Study 09: Which Classifier Should the Consortium Buy?

**DS701 — Session 11 (Tue Oct 13, 2026, Monday schedule) — Section A1**

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/class_activity_notebooks/09-InClass-Exercise-kNN-NB-A1/09-InClass-Exercise-kNN-NB-A1.ipynb)

**Time: ~45 minutes.** Work in groups of 2-3.

Parts marked **(autograded)** are submitted to Gradescope; open-ended parts are graded
for participation. You may use AI assistance, but you must be able to explain and
justify every part of your solution when asked.

**Plan**

| Part | The consortium's question | Method | Grading | Time |
|---|---|---|---|---|
| 1 | Does the $k$-NN vendor's model work on our measurements as they are, or must they be standardized first? | `StandardScaler` fitted on the training data; 5-NN before and after | autograded | ~7 min |
| 2 | How many neighbours, and how sure can we be of that number? | Sweep $k = 1\dots25$ under two metrics; 5-fold cross-validation | autograded | ~10 min |
| 3 | Head to head: $k$-NN, Naive Bayes, or the tree we already have? | Bake-off — confusion matrix, precision and recall on the malignant class | autograded | ~10 min |
| 4 | Can a pathologist follow Naive Bayes's arithmetic the way they can follow a tree? | Naive Bayes **by hand** on a 4-feature toy table, with and without smoothing | autograded | ~10 min |
| — | Report out: what you would tell the consortium | team memo, then pair-share | participation | ~10 min |

Dependencies: `numpy`, `pandas`, `matplotlib`, `scikit-learn`.


## Case Study Setup

You are the clinical research data scientist who built the Case Study 08 prototype for a
consortium of Boston research hospitals: **fine-needle aspirate (FNA)** biopsies, nucleus
measurements their imaging software already extracts, and a depth-limited decision tree
that flags likely-malignant samples for priority review. The pathologists liked the tree
— three questions they could read and check at the microscope. Now two vendors are
pitching the consortium a replacement: one sells a **$k$-nearest-neighbours** model, the
other a **Naive Bayes** model, and each claims to beat the tree. One of them also wants to
sell 500 extra texture features to go with it. The purchasing committee has asked you for
a recommendation: *which model, if any, should they buy — and on what condition?* What
hangs on the answer is the review queue: a model that misses malignant samples leaves them
waiting weeks for review, and a model nobody can explain will not be trusted by the
pathologists who have to act on it.

As in Case Study 08, before any of their own patients' data is released to you (that takes
IRB approval), they have asked you to run the comparison on the public **Wisconsin
Diagnostic Breast Cancer** dataset, collected at the University of Wisconsin Hospitals from
the same kind of FNA images. It ships with `scikit-learn`, so nothing is downloaded: 569
samples (212 malignant, 357 benign), 30 columns in three blocks of ten — the `mean`,
standard `error` and `worst` value of the same ten nucleus measurements. This time we work
with the **ten mean measurements** only. Part 4's e-mail table is a toy — twelve made-up
rows, not tumours — chosen so that every number in Naive Bayes's arithmetic can be checked
by hand.

**What the consortium asked you, and where you answer it**

1. *Does the $k$-NN vendor's model work on our measurements as they are, or must they be
   standardized first?* — Part 1
2. *How many neighbours should the model use, and how sure can we be of that number?* —
   Part 2
3. *Head to head: $k$-NN, Naive Bayes, or the tree we already have?* — Part 3
4. *Can a pathologist follow Naive Bayes's arithmetic the way they can follow a tree?* —
   Part 4

The vendor's 500 extra features come up at the report-out, where you answer from the
lecture rather than from code. A prototype for discussion, not a clinical tool: nothing
here is validated for patient care.

We call **malignant the positive class** (`y = 1`) — the one we must not miss. (Case Study
08 coded malignant as `0`; here the coding is flipped so that precision and recall read
naturally.) The split below is stratified 70/30 and fixed by the section seed. Every model
in Parts 1–3 is trained on `X_train` and judged on `X_test`.

Run the setup cells below as-is — everything after them depends on them.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix


In [ ]:
# Lecture section. Only A1 runs this semester; this is kept because the build
# script reads it to name the variant directory the notebook is published in.
SECTION = "A1"
SEED = 701
rng = np.random.default_rng(SEED)


In [ ]:
# Colab: fetch the autograder tests for this activity.
import os, sys, urllib.request

if "google.colab" in sys.modules:
    os.makedirs("tests", exist_ok=True)
    BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/class_activity_notebooks/09-InClass-Exercise-kNN-NB-A1/tests/"
    for _q in ("q1", "q2", "q3", "q4"):
        urllib.request.urlretrieve(BASE + _q + ".py", "tests/" + _q + ".py")


In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook()

In [ ]:
cancer = load_breast_cancer()

# The ten "mean ..." columns.
COLS = slice(0, 10)
feature_names = list(cancer.feature_names[COLS])
X_all = cancer.data[:, COLS]
y_all = (cancer.target == 0).astype(int)     # 1 = malignant (POSITIVE class), 0 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.3, stratify=y_all, random_state=SEED)
n_train, n_test = len(X_train), len(X_test)

print(f"Section {SECTION}: features = {feature_names}")
print(f"train {X_train.shape}, test {X_test.shape}; malignant in test: {int(y_test.sum())} of {n_test}")
pd.DataFrame(X_train, columns=feature_names).describe().loc[["mean", "std", "min", "max"]].round(3)


Look at the `std` row: some columns vary by hundreds, others by hundredths. Keep that in
mind for Part 1. Below, two of the features, coloured by class.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=18, alpha=0.8)
ax.set_xlabel(feature_names[0]); ax.set_ylabel(feature_names[1])
ax.set_title("training data (red = malignant)")
ax.grid(alpha=0.3)
plt.show()


## Part 1 (autograded): scale first, then $k$-NN

**Why.** The consortium's first question is whether the $k$-NN vendor's model can run on
the nucleus measurements as their imaging software exports them, or whether the
measurements must be standardized first — and, if so, whether that matters enough to write
into the contract. $k$-NN is built on Euclidean distance, so a column measured in hundreds
decides who the neighbours are all by itself. The fix is to standardise — but **fit the
scaler on the training data only** and apply the same transformation to the test data (the
test set must not leak into anything you fit). Measure what the difference is worth.

**Task.**

1. `scaler = StandardScaler().fit(X_train)`; `X_train_s`, `X_test_s` = the transformed
   training and test features.
2. Fit a 5-NN classifier on the **raw** features and one on the **scaled** features; store
   the test accuracies as `acc_raw` and `acc_scaled` (floats).


In [ ]:
scaler = ...
X_train_s = ...
X_test_s = ...

knn5_raw = ...
knn5_scaled = ...
acc_raw = ...
acc_scaled = ...

print(f"5-NN test accuracy  raw: {acc_raw:.4f}   scaled: {acc_scaled:.4f}")


**Discuss (participation):** the vendor's brochure says the model "uses all ten
measurements". Before scaling, which one or two columns were actually choosing the
neighbours (look at the `std` row above)? And is the measurement with the biggest spread
the one that tells a pathologist the most?

*(1-2 sentences here)*


In [ ]:
grader.check("q1")

## Part 2 (autograded): sweep $k$, choose it by cross-validation

**Why.** The vendor's model has a dial, $k$, and a choice of distance metric, and the
committee wants to know which setting to buy and how sure you are of it. Every $k$ is a
different model, and so is every distance metric. Rather than one train/test split (one
noisy sample of the accuracy), use **5-fold stratified cross-validation on the training
set** — the lecture averaged 50 random splits; $K$-fold CV is the tidier version from
lecture 07. The folds are fixed below so everyone gets the same numbers.

**Task.**

1. `cv_acc` — a dict `{metric: array}` where, for each metric in `METRICS`, the array holds
   the **mean CV accuracy** for `k = 1, 2, ..., 25` (in `K_VALUES` order), from
   `cross_val_score(KNeighborsClassifier(n_neighbors=k, metric=metric), X_train_s, y_train, cv=cv)`.
2. `best_metric`, `best_k` — the metric and $k$ with the highest mean CV accuracy
   (ties: the smaller $k$; if the two metrics tie, `"euclidean"`).
3. `knn_best` — a `KNeighborsClassifier(n_neighbors=best_k, metric=best_metric)` fitted on
   the scaled training data, and `acc_knn_test` — its test accuracy (float).


In [ ]:
K_VALUES = np.arange(1, 26)
METRICS = ["euclidean", "manhattan"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_acc = {}
...

print(f"best: metric = {best_metric}, k = {best_k}, CV accuracy = {cv_acc[best_metric].max():.4f}")
print(f"test accuracy of knn_best: {acc_knn_test:.4f}")


In [ ]:
grader.check("q2")

In [ ]:
plt.figure(figsize=(7, 4))
for m in METRICS:
    plt.plot(K_VALUES, cv_acc[m], ".-", label=m)
plt.axvline(best_k, color="gray", ls="--", lw=1)
plt.xlabel("k"); plt.ylabel("mean 5-fold CV accuracy"); plt.xticks(K_VALUES[::2])
plt.title(f"k-NN on scaled training data (chosen: {best_metric}, k = {best_k})")
plt.legend(); plt.grid(alpha=0.3)
plt.show()


<!-- BEGIN QUESTION -->

**Discuss (participation):** the committee wants a single value of $k$ written into the
purchase order. Where does $k = 1$ sit on your curve, and why would you refuse it? The
lecture's iris curve peaked and then fell; does yours? Differences of half a percent — with
~400 training patients and 5 folds, how many patients is that — so how firmly can you
specify "$k$ = `best_k`" rather than a range?

*(2-3 sentences here)*


_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Part 3 (autograded): the bake-off

**Why.** This is the question the committee actually has to decide: head to head, does
either vendor's model beat the depth-limited tree the pathologists already like? Three
classifiers, one test set. `knn_best` from Part 2, a **Gaussian Naive Bayes** (one
Gaussian per feature and class — the continuous-feature version of the counting you will
do by hand in Part 4), and a **decision tree** limited to depth 3 (Case Study 08's tool).
Accuracy is not enough for a review queue — a missed malignant sample waits weeks, a false
alarm costs one extra review — so read the confusion matrix and compute precision and
recall for the positive (malignant) class.

**Task.**

1. Complete `models`: `"nb"` = `GaussianNB()` and `"tree"` =
   `DecisionTreeClassifier(max_depth=3, random_state=SEED)`, both fitted on `X_train_s, y_train`.
2. For each model, predict on `X_test_s` and store `confusion_matrix(y_test, y_pred)` in
   `cms[name]` — rows are the actual class (0, 1), columns the predicted class.
3. From the confusion matrix (`tn, fp, fn, tp = cm.ravel()`) fill
   `results[name] = {"accuracy": ..., "precision": ..., "recall": ...}` with
   precision $= TP/(TP+FP)$ and recall $= TP/(TP+FN)$ for the malignant class.


In [ ]:
models = {
    "knn": knn_best,
    ...
    ...
}

cms = {}
results = {}
...

for name in models:
    print(f"{name:>5}: [[TN FP] [FN TP]] = {cms[name].tolist()}")
pd.DataFrame(results).T.round(3)


**Discuss (participation):** the committee will ask "which model is most accurate?" Which
number in your table would you tell them matters more for a review queue, and which model
wins on *that* number — is it the accuracy winner? Then: a pathologist asks each model "why
did you flag this sample?" What is the honest one-line answer for each of the three?

*(your answers here)*


In [ ]:
grader.check("q3")

## Part 4 (autograded): Naive Bayes by hand

**Why.** The pathologists liked the tree because they could follow it, and the Naive Bayes
vendor claims their model is just as readable — "it is only counting". Before you repeat
that to the consortium, do the counting yourself: what `GaussianNB` did in Part 3, but on
a toy table small enough that every number can be checked at the table. Twelve made-up
e-mails stand in for tumours — four categorical features and a label `spam`:

- `link` — contains a link (yes/no)
- `known_sender` — sender is in the address book (yes/no)
- `caps` — subject line in ALL CAPS (yes/no)
- `length` — short/long

For a query $\mathbf{x} = (x_1,\dots,x_4)$ Naive Bayes scores each class $c$ as

$$\text{score}(c) = P(c)\prod_{j=1}^{4} P(x_j \mid c),$$

with $P(c)$ = fraction of rows in class $c$ and $P(x_j = v \mid c)$ = fraction of class-$c$
rows with feature $j$ equal to $v$. With **Laplace smoothing** $\alpha$ the likelihoods become
$\dfrac{\#\{x_j = v,\ c\} + \alpha}{\#\{c\} + \alpha\, V_j}$, where $V_j$ is the number of
distinct values feature $j$ takes in the whole table (leave the prior unsmoothed).

**Task.**

1. Write `nb_scores(df, query, target, alpha=0)` returning `{class_value: score}` for every
   class value in `df[target]`, where `query` is a dict `{feature: value}`.
2. `scores_raw = nb_scores(emails, QUERY, "spam", alpha=0)` and
   `scores_smooth = nb_scores(emails, QUERY, "spam", alpha=1)`.
3. `nb_prediction` — the class with the larger **smoothed** score, and `p_spam_smooth` —
   the smoothed posterior $P(\text{spam}=\text{yes} \mid \mathbf{x})$, i.e. the "yes" score divided by
   the sum of both smoothed scores.


In [ ]:
emails = pd.DataFrame({
    "link":         ["yes", "yes", "yes", "no",  "yes", "no",  "no",  "yes", "no",  "no",  "yes", "no"],
    "known_sender": ["no",  "no",  "no",  "no",  "no",  "yes", "yes", "yes", "no",  "yes", "no",  "yes"],
    "caps":         ["yes", "yes", "no",  "yes", "no",  "no",  "no",  "no",  "no",  "no",  "no",  "no"],
    "length":       ["short","long","short","long","long","long","short","long","short","long","short","short"],
    "spam":         ["yes", "yes", "yes", "yes", "yes", "no",  "no",  "no",  "no",  "no",  "no",  "no"],
})
QUERY = {"link": "yes", "known_sender": "yes", "caps": "no",  "length": "short"}
print("query for section", SECTION, ":", QUERY)
display(emails)


def nb_scores(df, query, target, alpha=0):
    """Naive Bayes class scores P(c) * prod_j P(x_j | c) from counts, with Laplace smoothing alpha."""
    scores = {}
    ...
    return scores


scores_raw = ...
scores_smooth = ...
nb_prediction = ...
p_spam_smooth = ...

print("raw scores     :", {c: round(v, 5) for c, v in scores_raw.items()})
print("smoothed scores:", {c: round(v, 5) for c, v in scores_smooth.items()})
print(f"prediction: spam = {nb_prediction};  P(spam | x) = {p_spam_smooth:.3f}")


**Discuss (participation):** suppose this table were the consortium's biopsies. What did
the zero in `scores_raw` do to the decision, and would you let a model set a sample's
priority on one empty count? Did smoothing change the *prediction* or only the
*confidence*? Then, in one sentence for a pathologist: what is the same between this
arithmetic and `GaussianNB` in Part 3, and what is different?

*(2-3 sentences here)*


In [ ]:
grader.check("q4")

<!-- BEGIN QUESTION -->

## Report out: what you would tell the consortium

Write the two sentences you would put at the top of your memo to the consortium: the
recommendation — which model, if any, they should buy — and the one caveat they must hear.
Cite numbers from your notebook. Then pair-share with another team: did you recommend the
same model? If not, whose evidence is stronger?

Discuss as a team before you write:

1. **The one condition on the purchase.** Standardisation (Part 1), the value of $k$ and
   how firmly you would specify it (Part 2), recall on malignant rather than accuracy
   (Part 3) — which of these goes into the contract, and in what words?
2. **The vendor's 500 extra texture features.** One vendor wants to add 500 texture
   features to the ten you used. From the lecture — distances concentrate, and the fraction
   of points in the outer shell $1-(1-\epsilon)^d$ goes to 1 — what happens to the word
   "nearest" when $d$ goes from 10 to 510, and does Naive Bayes suffer the same way? Answer
   from the lecture; no code.
3. **Readability.** Which of the three could a pathologist follow (Parts 3 and 4), and
   does that change which one you recommend?

> Staff will visit tables during this part. Be ready to explain your reasoning and to
> point at the cell that supports each claim.

**Submit.** Run all cells top to bottom, confirm every `grader.check(...)` cell passes,
and submit this notebook to the Gradescope assignment **Activity 09 (Section A1)**.


_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Wrap-up

Before you leave, each group should be able to answer, without notes:

1. Why must the scaler be fitted on the training data only, and what breaks for $k$-NN
   without scaling at all?
2. Why did you choose $k$ by cross-validation on the training set rather than on the test
   set, and how firmly does your curve let you specify it?
3. Which number in the bake-off table matters most for a review queue, and which model won
   it?
4. In one sentence: what does the naive assumption buy Naive Bayes when a vendor adds
   hundreds of features?

Submit the autograded parts (1–4) to Gradescope.
